In [1]:
import pandas as pd

# Load data
df = pd.read_csv("../data/processed/window_features.csv")

print("Original shape:", df.shape)
print(df.head())

Original shape: (7299, 20)
  meter_id  window_id      mean       std      min       max    median  \
0     BR02          0  0.006529  0.007243  0.00035  0.027600  0.002825   
1     BR02          1  0.000913  0.003028  0.00000  0.011170  0.000000   
2     BR02          2  0.004579  0.005430  0.00000  0.020250  0.001650   
3     BR02          3  0.000101  0.000333  0.00000  0.001209  0.000000   
4     BR02          4  0.000000  0.000000  0.00000  0.000000  0.000000   

      range  load_factor  peak_to_avg        cv  mean_abs_diff  std_diff  \
0  0.027250     0.236564     4.227186  1.109299       0.007322  0.010007   
1  0.011170     0.081729    12.235509  3.317322       0.000486  0.002188   
2  0.020250     0.226132     4.422202  1.185853       0.003646  0.005730   
3  0.001209     0.083183    12.021666  3.316631       0.000053  0.000247   
4  0.000000     0.000000     0.000000  0.000000       0.000000  0.000000   

   day_mean  night_mean  day_night_ratio  zero_pct   low_pct  max_low_s

In [ ]:
#filter ..only keep normal data not theft data

In [4]:
# Keep only normal (non-theft)
df_normal = df[df["theft"] == 0].copy()

print("Normal-only shape:", df_normal.shape)
print("\nTheft distribution after filter:")
print(df_normal["theft"].value_counts())

Normal-only shape: (6610, 20)

Theft distribution after filter:
theft
0.0    6610
Name: count, dtype: int64


In [5]:
# Drop non-feature columns
drop_cols = ["meter_id", "window_id", "theft"]

X_normal = df_normal.drop(columns=drop_cols)

print("\nFeature matrix shape:", X_normal.shape)
print("\nColumns:")
print(X_normal.columns.tolist())


Feature matrix shape: (6610, 17)

Columns:
['mean', 'std', 'min', 'max', 'median', 'range', 'load_factor', 'peak_to_avg', 'cv', 'mean_abs_diff', 'std_diff', 'day_mean', 'night_mean', 'day_night_ratio', 'zero_pct', 'low_pct', 'max_low_streak']


In [ ]:
# Verification (Passed)
# 🔹 Data Split Integrity
# Total rows: 7299
# Normal rows: 6610
# Theft rows: ~689 (~9.4%)

# ✔ Matches your simulation design (~9%)
# ✔ No leakage into anomaly training

# 🔹 Feature Matrix Check
# Shape: (6610, 17) ✅
# Columns: all engineered features only ✅
# Dropped:
# meter_id ✔
# window_id ✔
# theft ✔
# 🔹 Feature Types (Critical for anomaly models)

# Your features include:

# Statistical → mean, std, min, max, median, range
# Behavioral → load_factor, peak_to_avg, cv
# Dynamics → mean_abs_diff, std_diff
# Temporal → day/night features
# Theft-sensitive → zero_pct, low_pct, max_low_streak

# ✔ This is very strong for anomaly detection — especially:

# zero_pct
# low_pct
# max_low_streak
# cv
# peak_to_avg

# These will drive anomaly separation.

In [ ]:
# Step 2: Feature Scaling (CRITICAL)
#  Why scaling is mandatory
# LOF = distance-based → MUST scale
# Isolation Forest = tree-based → scaling optional but we standardize for consistency

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_normal)

print("Scaled shape:", X_scaled.shape)
print("Sample (first 2 rows):")
print(X_scaled[:2])

Scaled shape: (6610, 17)
Sample (first 2 rows):
[[-0.75524355 -0.42477142 -0.68339022 -0.50938003 -0.84130281 -0.42960496
  -1.07645611  0.76550005  0.84316391 -0.11035795 -0.19639698 -0.68796804
  -0.74288511  0.01279628 -0.39103341  1.39372081  0.83560503]
 [-1.04902772 -0.76817252 -0.7385804  -0.88337385 -0.99388006 -0.81245338
  -1.89436676  4.67512146  4.9178248  -0.94552426 -0.86745506 -1.09529993
  -0.9186562  -0.11562018  3.46913075  4.78262521  5.26387035]]


In [8]:
#Save scaler (VERY IMPORTANT for inference later)
import joblib

joblib.dump(scaler, "models/scaler_anomaly.pkl")

print("Scaler saved ✅")

Scaler saved ✅


In [ ]:
# Verification (Passed)
# 🔹 Shape Check
# (6610, 17) → unchanged ✔
# 🔹 Distribution Check (important)

# From your sample:

# Values are centered around 0
# Spread across positive/negative → proper standardization
# No NaNs / inf → ✔
# 🔹 Key Observation

# Second row has very high z-scores:

# peak_to_avg ≈ 4.67
# cv ≈ 4.91
# low_pct ≈ 4.78
# max_low_streak ≈ 5.26

# 👉 This is exactly what we want:
# These represent extreme behavior patterns, which anomaly models will detect.

In [ ]:
# Isolation Forest

# Now we train the first anomaly model.

# 🎯 Key Design Decisions (VERY IMPORTANT)
# 🔹 Why Isolation Forest first?
# Works well on high-dimensional tabular data
# Robust to noise
# Captures global anomalies
# 🔹 Contamination (CRITICAL PARAMETER)

# We trained only on normal data, BUT:

# 👉 Real system assumption:

# Some unseen anomalies may exist
# We simulate detection sensitivity
# ✅ Set:
# contamination = 0.05

# This means:

# Model will flag ~5% most abnormal patterns

In [9]:
#Train Isolation Forest
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

iso_model.fit(X_scaled)

print("Isolation Forest trained ✅")

Isolation Forest trained ✅


In [10]:
#Generate anomaly predictions (on normal data
iso_preds = iso_model.predict(X_scaled)

# Convert to readable format
# -1 = anomaly, 1 = normal
import numpy as np

unique, counts = np.unique(iso_preds, return_counts=True)

print("Prediction distribution:")
for u, c in zip(unique, counts):
    label = "Anomaly (-1)" if u == -1 else "Normal (1)"
    print(f"{label}: {c}")

Prediction distribution:
Anomaly (-1): 331
Normal (1): 6279


In [11]:
import joblib

joblib.dump(iso_model, "models/isolation_forest.pkl")

print("Isolation Forest saved ✅")

Isolation Forest saved ✅


In [ ]:
# Verification (Isolation Forest)
# 🔹 Distribution Check
# Total: 6610
# Anomalies: 331 (~5.0%)
# Normal: 6279

# ✔ Matches contamination=0.05 perfectly
# ✔ Model is neither collapsing nor over-flagging

# 🔹 Interpretation (Important for your system design)

# Even though training data = only normal, the model still flags:

# 👉 “statistical extremes within normal behavior”

# These include:

# Very high cv
# Extreme peak_to_avg
# High low_pct / long low streaks
# Flat or zero-heavy windows

# ⚠️ This is expected and desired
# → These become "proto-anomalies"

# Later in ensemble:

# If supervised ALSO agrees → HIGH RISK
# If not → SUSPICIOUS
# 🔹 Key Insight (Architecture-level)

# You now have:

# Isolation Forest → global anomaly detector
# Will catch:
# Distribution-level deviations
# Extreme consumption patterns

# But it may miss:

# Subtle local deviations within clusters

# 👉 That’s exactly why we add LOF next

In [ ]:
# Local Outlier Factor (LOF)
# 🎯 Purpose

# Detect local anomalies:

# Points that look normal globally
# But abnormal relative to neighbors
# ⚠️ Important Difference from Isolation Forest

# LOF:

# Is distance-based
# Sensitive to local density
# Needs scaled data ✔ (already done)
# 🔹 Key Parameter: n_neighbors

# We choose:

# n_neighbors = 20

# Reason:

# Your dataset is moderately large (6610)
# 20 captures local neighborhood structure without overfitting

In [13]:
#Train LOF
from sklearn.neighbors import LocalOutlierFactor

lof_model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    n_jobs=-1
)

lof_preds = lof_model.fit_predict(X_scaled)

print("LOF trained and predictions generated ✅")

LOF trained and predictions generated ✅


In [14]:
#Check prediction distribution
import numpy as np

unique, counts = np.unique(lof_preds, return_counts=True)

print("LOF Prediction distribution:")
for u, c in zip(unique, counts):
    label = "Anomaly (-1)" if u == -1 else "Normal (1)"
    print(f"{label}: {c}")

LOF Prediction distribution:
Anomaly (-1): 331
Normal (1): 6279


In [ ]:
# Verification (LOF)
# 🔹 Distribution
# Anomalies: 331 (~5%)
# Normal: 6279

# ✔ Matches contamination
# ✔ No instability
# ✔ No overfitting behavior

# 🔹 Interpretation (Important)

# Even though counts match Isolation Forest:

# 👉 They are NOT the same points

# Isolation Forest → global deviations
# LOF → local density deviations

# This distinction is critical for your hybrid system.

# 🔹 What LOF is likely catching

# Compared to Isolation Forest, LOF is better at:

# Subtle behavioral deviations within similar meters
# Slight but consistent drops
# Pattern shifts not extreme globally

# Example (in your features):

# Slight increase in low_pct vs neighbors
# Slight abnormal day_night_ratio
# Mild but consistent mean_abs_diff shifts
# 🔥 Key System Insight (Very Important)

# You now have:

# Model	Strength
# Isolation Forest	Global anomalies
# LOF	Local anomalies

# 👉 This is exactly why your system is stronger than standard anomaly detection

In [ ]:
# Final LOF Model

# rebuild LOF for Inference (CRITICAL)

# As mentioned:

# ⚠️ Current LOF cannot be used on new data

# We must rebuild with:

# novelty=True

In [15]:
from sklearn.neighbors import LocalOutlierFactor

lof_model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=True,   # IMPORTANT
    n_jobs=-1
)

lof_model.fit(X_scaled)

print("LOF (novelty mode) trained ✅")

LOF (novelty mode) trained ✅


In [17]:
#Test prediction (sanity check)
lof_preds_new = lof_model.predict(X_scaled)

import numpy as np

unique, counts = np.unique(lof_preds_new, return_counts=True)

print("LOF (novelty mode) distribution:")
for u, c in zip(unique, counts):
    label = "Anomaly (-1)" if u == -1 else "Normal (1)"
    print(f"{label}: {c}")

LOF (novelty mode) distribution:
Anomaly (-1): 295
Normal (1): 6315


In [19]:
import joblib

joblib.dump(lof_model, "models/lof_model.pkl")

print("LOF model saved ✅")

LOF model saved ✅


In [ ]:
# Now your anomaly layer behaves like this:

# Isolation Forest
# Fixed anomaly rate (~5%)
# More aggressive
# Captures global extremes
# LOF (novelty)
# Slightly conservative (~4.5%)
# Captures local deviations
# Reduces noise
# 🧠 Combined Effect

# This is ideal for your hybrid system:

# IF → ensures anomalies are not missed
# LOF → ensures anomalies are meaningful

# 👉 Together:
# High recall + controlled precision